In [1]:
'''
您是一位經驗豐富的新聞記者，負責根據引用的事實資訊撰寫客觀的新聞文章。您的職責是：
1. 使用引用的事實撰寫全面的新聞報導
2. 運用新聞寫作技巧自然地連接資訊（何人、何事、何時、何地、為何、如何）
3. 保持嚴格的事實準確性 - 不推測或添加超出引述範圍的細節
4. 在保留引述中所有重要細節的同時，讓文章結構具有邏輯性
5. 將輸出內容置於 ###輸出開始### 和 ###輸出結束### 標記之間
指導方針：
* 僅使用編號引述明確支持的資訊
* 使用自然的過渡同時保持準確性
* 應用標準新聞寫作風格和結構
* 包含引述中的所有相關事實
* 避免任何推測或未經支持的細節


這裡是一些範例參考撰寫方式，請學習這些範例來撰寫你的新聞報導
{examples}

請根據以下引述資訊撰寫新聞文章：
{input_data}

請以以下格式回答：
###輸出開始###
[你的回答]
###輸出結束###
'''

'\n您是一位經驗豐富的新聞記者，負責根據引用的事實資訊撰寫客觀的新聞文章。您的職責是：\n1. 使用引用的事實撰寫全面的新聞報導\n2. 運用新聞寫作技巧自然地連接資訊（何人、何事、何時、何地、為何、如何）\n3. 保持嚴格的事實準確性 - 不推測或添加超出引述範圍的細節\n4. 在保留引述中所有重要細節的同時，讓文章結構具有邏輯性\n5. 將輸出內容置於 ###輸出開始### 和 ###輸出結束### 標記之間\n指導方針：\n* 僅使用編號引述明確支持的資訊\n* 使用自然的過渡同時保持準確性\n* 應用標準新聞寫作風格和結構\n* 包含引述中的所有相關事實\n* 避免任何推測或未經支持的細節\n\n\n這裡是一些範例參考撰寫方式，請學習這些範例來撰寫你的新聞報導\n{examples}\n\n請根據以下引述資訊撰寫新聞文章：\n{input_data}\n\n請以以下格式回答：\n###輸出開始###\n[你的回答]\n###輸出結束###\n'

In [14]:
from typing import Dict, List, Any, Tuple
import json
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain.schema import Document
from utils.api_fetcher import fetch_data_and_save
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Constants
CHUNK_SIZE: int = 100
CHUNK_OVERLAP: int = 10

In [15]:
def get_project_paths() -> Tuple[str, str]:
    """Get current directory and project root paths."""
    current_dir: str = os.getcwd()
    project_root: str = os.path.join(current_dir, os.pardir)
    return current_dir, project_root


def load_json_data(project_root: str) -> Dict[str, List]:
    """Load JSON data from file."""
    json_file_path: str = os.path.join(project_root, 'data', 'example_api_data.json')
    with open(json_file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
# Global variables


In [41]:
current_dir, project_root = get_project_paths()
data = load_json_data(project_root)

# Load environment variables
dotenv_path: str = os.path.join(project_root, 'config', '.env')
load_dotenv(dotenv_path)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

original_id_to_new_id: Dict[str, str] = {}
new_id_to_original_id: Dict[str, str] = {}

In [17]:
print(data)

{'content': [{'id': 'd6202d36-3705-4fe7-bef5-0929f52ea1e1', 'createdAt': '2024-12-05T22:04:05.502525', 'updatedAt': '2024-12-05T22:04:05.502525', 'title': '售票系統使用起來體感不佳', 'content': '可惡，網路不夠快，導致我完全搶不到票\n要買其他人的讓票，一張票還被炒到3倍價格', 'authorId': 'ca7885d9-2c42-4418-b9a8-dabfab4afa02', 'authorName': '小知', 'authorAvatar': '/api/user/avatar/yukina', 'likeCount': 1, 'reasonableCount': 0, 'dislikeCount': 0, 'userReaction': {'reaction': 'LIKE'}, 'facts': [{'id': '738851aa-018a-4db8-8f0f-74c45b32b96e', 'createdAt': '2024-12-05T22:01:21.162607', 'updatedAt': '2024-12-05T22:01:21.162607', 'title': '假票券', 'authorId': 'ca7885d9-2c42.-4418-b9a8-dabfab4afa02', 'authorName': '小知', 'authorAvatar': '/api/user/avatar/yukina', 'references': [{'id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'url': 'https%3A%2F%2Fwww.nownews.com%2Fnews%2F6599071', 'icon': 'https%3A%2F%2Fwww.nownews.comhttps%3A%2F%2Fmedia.nownews.com%2Fnn_media%2Fthumbnail%2F2024%2F10%2F1729259015127-5ecd1dc0aad14918a429ab02b7ad816b-1200x798.webp%3F

In [18]:
def process_json(data: Dict[str, List]) -> str:
    '''format data'''
    result: str = ""
    now_id: int = 1
    for comment in data["content"]:
        fact_num: int = 1
        for fact in comment["facts"]:
            result += f"事實{fact_num}:\n\n"
            for reference in fact["references"]:
                if reference["id"] not in original_id_to_new_id:
                    
                    text: str = f"{reference['title']}。{reference['description']}"
                    result += f"參考資料:\n{text}[{now_id}]\n\n"
                    original_id_to_new_id[reference["id"]] = str(now_id)
                    new_id_to_original_id[str(now_id)] = reference["id"]
                    now_id += 1
            fact_num += 1
    return result

In [19]:
def get_user_message_imply(
    llm: ChatOpenAI,
    user_message: str,
    html_metadata: str
) -> Any:
    """Get implied message from user input."""
    prompt = PromptTemplate.from_template(
        '''
        你是一個能夠閱讀文章描述與使用者留言的助手。請根據下面的「文章描述」與「使用者評論」，
        推斷這篇文章中尚未明確提到但很可能被涵蓋的內容，並寫成簡潔的摘要。

        文章描述:
        {html_metadata}

        使用者評論:
        {user_message}

        請將可能被涵蓋的內容以一段話敘述出來，並用中文回答。
        '''
    )
    result = prompt | llm
    return result.invoke({
        "user_message": user_message,
        "html_metadata": html_metadata
    })

In [20]:
def process_message(
    data: Dict[str, Any],
    llm: ChatOpenAI
) -> str:
    """
    對每個 comment 的 user_message 與 reference 做推斷，
    並回傳類似 process_json 的格式化字串。
    """
    result: str = ""
    comment_num: int = 1

    for comment in data["content"]:
        # 1. 取得使用者留言
        user_message: str = comment["content"]

        # 2. 收集對應的 reference，組合成一大段文字 (html_metadata)
        references_text_list = []
        for fact in comment["facts"]:
            for reference in fact["references"]:
                
                html_metadata = f"{reference['title']}。{reference['description']}"
                
                # 3. 呼叫 get_user_message_imply，取得推斷內容
                #    注意 get_user_message_imply(...) 回傳的是一個 chain 後的結果對象，你可能需要 .content
                #    或者如果你已經在 get_user_message_imply 裏面有 .invoke(...)，它會回傳 AIMessage 或類似物件
                imply_result = get_user_message_imply(llm, user_message, html_metadata)

                # 有些時候 imply_result 可能是 AIMessage 或字串，請視你的 get_user_message_imply 實作調整
                # 這裡假設 imply_result 是 chain invoke 後的「AIMessage」
                implied_text = imply_result.content if hasattr(imply_result, "content") else str(imply_result)

                # 4. 依需求把資訊組成 result 字串
                #    - 這裡示範做法：我們先列出「留言X」再列出原文，再列出推斷
                result += f"留言 {comment_num}:\n"
                result += f"原始留言：\n{user_message}\n\n"
                result += f"推斷內容：\n{implied_text}[U{comment_num}]\n\n"
                comment_num += 1

    return result


In [42]:
html_metadata_string = process_json(data)
user_imply_message_string = process_message(data, llm)
model_input_string = html_metadata_string + user_imply_message_string

# Load few-shot example
file_path: str = os.path.join(project_root, 'data', 'few_shot_cot.txt')
with open(file_path, 'r', encoding="utf-8") as file:
    few_shot_example: str = file.read()

In [43]:
html_metadata_string

'事實1:\n\n參考資料:\n周杰倫演唱會「黃牛票」何時釋出？拓元低調回應 歌迷再度崩潰了 | 娛樂 | NOWnews今日新聞。這篇文章說明了周杰倫演唱會黃牛票的問題，並提到拓元的官方回應。[1]\n\n事實2:\n\n參考資料:\n日本演唱會抽選制，為何台灣不行？ - 商周。article description[2]\n\n'

In [44]:
print(model_input_string)

事實1:

參考資料:
周杰倫演唱會「黃牛票」何時釋出？拓元低調回應 歌迷再度崩潰了 | 娛樂 | NOWnews今日新聞。這篇文章說明了周杰倫演唱會黃牛票的問題，並提到拓元的官方回應。[1]

事實2:

參考資料:
日本演唱會抽選制，為何台灣不行？ - 商周。article description[2]

留言 1:
原始留言：
可惡，網路不夠快，導致我完全搶不到票
要買其他人的讓票，一張票還被炒到3倍價格

推斷內容：
這篇文章可能還涵蓋了周杰倫演唱會門票供應不足的情況，導致歌迷搶票困難，並且可能提到轉售市場的問題，例如黃牛票的價格炒高，讓原本想要參加演唱會的粉絲感到沮喪和無奈。此外，文章可能也涉及對於拓元的回應是否能有效解決這些問題的討論。[U1]
留言 2:
原始留言：
可惡，網路不夠快，導致我完全搶不到票
要買其他人的讓票，一張票還被炒到3倍價格

推斷內容：
這篇文章可能探討了日本演唱會的票務抽選制度如何運作，以及其在台灣推行的可行性和挑戰。此外，文章或許還會涉及票務搶購的困難，例如網路速度對購票成功的影響，以及轉售市場中票價炒作的問題，反映出粉絲在購票過程中面臨的困境與不公平現象。[U2]



In [ ]:
#COT prompting 7. 學習範例中的思考步驟，先經過和思考後，再給出引用答案

In [45]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate([
    ("system", '''
您是一位經驗豐富的新聞記者，負責根據引用的事實資訊撰寫客觀的新聞文章。您的職責是：
1. 使用引用的事實撰寫全面的新聞報導
2. 運用新聞寫作技巧自然地連接資訊（何人、何事、何時、何地、為何、如何）
3. 保持嚴格的事實準確性 - 不推測或添加超出引述範圍的細節
4. 在保留引述中所有重要細節的同時，讓文章結構具有邏輯性
5. 將輸出內容置於 ###輸出開始### 和 ###輸出結束### 標記之間
6. 有兩種引用格式分別是 [數字] 例如 [1] [2] 或者是 [數字U] [U1] [U2]
7. 學習範例中的思考步驟，先經過和思考後，再給出引報導內容，再思考的文字之前統一先輸出(事實綜合整理和思考：)
指導方針：
* 僅使用編號引述明確支持的資訊
* 使用自然的過渡同時保持準確性
* 應用標準新聞寫作風格和結構
* 包含引述中的所有相關事實
* 避免任何推測或未經支持的細節
* 必須先經過和思考後，再給出報導內容


這裡是一些範例參考撰寫方式，請學習這些範例來撰寫你的新聞報導
{examples}
     
請以以下格式回答：
###輸出開始###
[你的回答]
###輸出結束###'''
),
    ("human",'''請根據以下引述資訊撰寫新聞文章：
{input_data}'''),
])



In [48]:
result.content

'###輸出開始###\n事實綜合整理和思考：\n根據 [1]，周杰倫演唱會的黃牛票問題引發歌迷不滿，拓元的官方回應也未能平息情緒。根據 [2]，文章探討了台灣與日本在演唱會票務抽選制的差異，並提到票務搶購的困難及轉售市場的問題。透過留言可見，歌迷對於搶票困難感到沮喪，並且面臨高價黃牛票的困擾。\n\n報導內容：\n周杰倫的演唱會票務問題再次成為焦點，許多歌迷在搶票過程中遭遇困難，導致部分票務轉售價格飆升至原價的三倍，讓粉絲感到無奈和沮喪。[1] 對於黃牛票的問題，拓元官方的回應並未有效解決歌迷的焦慮，反而引發了更大的不滿情緒。\n\n此外，針對票務搶購的挑戰，文章也提到日本演唱會的抽選制，並討論了該制度在台灣的可行性。[2] 歌迷在留言中表達對於網路速度的擔憂，認為這影響了購票的成功率，使得原本想參加演出的人士面臨不公的競爭環境。\n\n整體來看，周杰倫演唱會的票務問題不僅反映出粉絲的期待與失落，也突顯了台灣票務市場在管理與公平性方面亟需改進的現狀。###輸出結束###'

In [47]:
chain = template | llm
result = chain.invoke({
    "input_data": model_input_string,
    "examples": few_shot_example
})

# Process and save final output
result_content: str = result.content[11:-11]
result_content = result_content.replace("\n", "")
final_output: Dict[str, Any] = {
    "Summary": result_content,
    "Citations": {
        str(citation): new_id_to_original_id[str(citation)]
        for citation in sorted(set(
            map(int, re.findall(r'\[(\d+)\]', result_content) +
                re.findall(r'\[U(\d+)\]', result_content))
        ))
    }
}


In [38]:
result_content

'周杰倫的演唱會票務問題引發歌迷廣泛關注，尤其是「黃牛票」現象造成許多粉絲無法以正當價格購得票券。根據報導，拓元對於黃牛票的問題發表了低調回應，但這並未減輕歌迷的失望情緒，許多粉絲在網路上表達了對於搶票困難的無奈與不滿，甚至有留言指出因為網路速度不夠快，導致他們完全搶不到票，並且在二手市場上，票價被炒至三倍的價格。[1]此外，針對票務系統的有效性，專家也討論了日本演唱會的抽選制，並提出該制度在票務分配上所具備的優勢，特別是如何減少搶票的競爭壓力。這引發了對於台灣是否應該考慮採用類似系統的討論，因為目前的票務系統存在著網路速度影響搶票結果及票價炒高等問題，讓許多歌迷感到困惑與不公平。[2] 這一系列事件表明，周杰倫演唱會的票務市場面臨著不少挑戰，如何有效解決黃牛票問題並改善票務分配系統，將成為未來討論的重點。 '

In [19]:
print(final_output)

{'Summary': '報導內容：周杰倫的演唱會近期引發「黃牛票」的問題，讓許多歌迷感到沮喪。根據報導，拓元對於黃牛票的問題作出低調回應，卻未能平息歌迷的怒火，許多粉絲對此再度感到崩潰。[1] 此外，報導也提到，與日本演唱會採取抽選制的做法相比，台灣目前尚未採用類似的票務管理措施，引發了對於票務公平性的討論。[2] ', 'Citations': {'1': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', '2': '556909b4-6a18-4390-bfff-87885d49e285'}}
